<!-- paired-role-banner -->
> 🟧 **실습 노트북 07 · 빈 코드 창**
>
> 오른쪽의 같은 이름 **정답 노트북**과 셀 위치를 맞춰 보세요. 먼저 정답을 가리고 직접
> 작성한 뒤 막힐 때 한 줄씩 비교하는 방식을 권장합니다. `NotImplementedError`가 있는 셀을
> 완성하면 됩니다.

# 07. Neural Networks from Scratch

PyTorch의 고수준 Trainer 없이 **손실 함수, 선형/로지스틱 회귀, MLP 학습**을 직접 구성합니다. 수식을 tensor 연산으로 옮기는 감각과 모델이 학습되지 않을 때 확인할 기준을 만듭니다.

예상 시간: 90~120분 · 실행 장치: CUDA → MPS → CPU 자동 선택 · 모든 데이터는 셀에서 합성

## 학습 지도

1. MSE와 선형 회귀: forward/backward/step의 최소 단위
2. sigmoid와 binary cross entropy: 확률적 이진 분류
3. MLP와 비선형성: 선형 모델이 못 푸는 XOR
4. activation과 initialization: 신호·gradient의 흐름
5. dropout과 normalization: 학습/추론 동작 차이
6. 작은 배치 과적합: 가장 강력한 학습 파이프라인 점검법

In [ ]:
import math
import pandas as pd
import torch
from torch import nn
from llm_engineering_lab.acceleration import get_accelerator

SEED = 42
torch.manual_seed(SEED)
torch.set_printoptions(precision=4, sci_mode=False)
ACCELERATOR = get_accelerator()
device = ACCELERATOR.device
torch.set_default_device(device)

print("torch:", torch.__version__)
print(ACCELERATOR.summary())

## 1. MSE와 선형 회귀

회귀에서는 예측 $ŷ$과 정답 $y$의 거리를 줄입니다.

$$\mathrm{MSE}(y,\hat{y}) = \frac{1}{N}\sum_{i=1}^{N}(y_i-\hat{y}_i)^2$$

작은 toy data에서는 정답 관계를 알고 있으므로 학습 결과를 검산할 수 있습니다.

In [ ]:
x_reg = torch.linspace(-2, 2, 64).unsqueeze(1)
noise = 0.15 * torch.randn(x_reg.shape)
y_reg = 3.0 * x_reg - 0.5 + noise

print("x/y shape:", x_reg.shape, y_reg.shape)
print(torch.cat([x_reg[:3], y_reg[:3]], dim=1))
assert x_reg.shape == y_reg.shape == (64, 1)

In [ ]:
# TODO 1: 1. MSE와 선형 회귀
# 오른쪽 정답 창의 같은 셀을 참고하되, 먼저 아래 구현을 직접 작성하세요.
# 1) 입력과 출력의 shape/dtype을 종이에 적습니다.
# 2) 최소 구현을 작성하고 Shift+Enter로 실행합니다.
# 3) 정답과 비교한 뒤 값 하나를 바꾸고 결과를 설명합니다.
raise NotImplementedError("07번 실습의 TODO 1을 완성하세요")

In [ ]:
# nn.Linear 없이 parameter 두 개로 y_hat = xw + b를 구현합니다.
torch.manual_seed(SEED)
weight = torch.randn(1, 1, requires_grad=True)
bias = torch.zeros(1, requires_grad=True)
learning_rate = 0.08

for step in range(151):
    prediction = x_reg @ weight + bias
    loss = mse_manual(prediction, y_reg)
    loss.backward()

    with torch.no_grad():
        weight -= learning_rate * weight.grad
        bias -= learning_rate * bias.grad
    weight.grad = None
    bias.grad = None

    if step % 50 == 0:
        print(f"step={step:3d} loss={loss.item():.5f}")

print("learned weight/bias:", weight.item(), bias.item())
assert abs(weight.item() - 3.0) < 0.1
assert abs(bias.item() + 0.5) < 0.1

In [ ]:
# 같은 계산을 nn.Linear + optimizer로 다시 표현합니다.
torch.manual_seed(SEED)
linear_model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(linear_model.parameters(), lr=0.08)

for _ in range(151):
    optimizer.zero_grad(set_to_none=True)
    loss = nn.functional.mse_loss(linear_model(x_reg), y_reg)
    loss.backward()
    optimizer.step()

print("nn.Linear weight/bias:", linear_model.weight.item(), linear_model.bias.item())
print("parameter names:", [name for name, _ in linear_model.named_parameters()])
assert linear_model(x_reg).shape == y_reg.shape

## 2. Logistic regression

선형 점수(logit)를 sigmoid로 확률로 바꿉니다.

$$z=xW+b, \qquad p(y=1|x)=\sigma(z)=\frac{1}{1+e^{-z}}$$

학습에는 확률을 먼저 만든 뒤 log를 취하기보다 `binary_cross_entropy_with_logits`를 쓰는 것이 수치적으로 안정적입니다.

In [ ]:
torch.manual_seed(SEED)
class0 = torch.randn(60, 2) * 0.45 + torch.tensor([-1.0, -0.8])
class1 = torch.randn(60, 2) * 0.45 + torch.tensor([1.0, 0.8])
x_binary = torch.cat([class0, class1])
y_binary = torch.cat([torch.zeros(60), torch.ones(60)]).unsqueeze(1)

permutation = torch.randperm(len(x_binary))
x_binary, y_binary = x_binary[permutation], y_binary[permutation]
print("features/labels:", x_binary.shape, y_binary.shape)
print("positive rate:", y_binary.mean().item())

In [ ]:
# TODO 2: 2. Logistic regression
# 오른쪽 정답 창의 같은 셀을 참고하되, 먼저 아래 구현을 직접 작성하세요.
# 1) 입력과 출력의 shape/dtype을 종이에 적습니다.
# 2) 최소 구현을 작성하고 Shift+Enter로 실행합니다.
# 3) 정답과 비교한 뒤 값 하나를 바꾸고 결과를 설명합니다.
raise NotImplementedError("07번 실습의 TODO 2을 완성하세요")

In [ ]:
# TODO 3: 2. Logistic regression
# 오른쪽 정답 창의 같은 셀을 참고하되, 먼저 아래 구현을 직접 작성하세요.
# 1) 입력과 출력의 shape/dtype을 종이에 적습니다.
# 2) 최소 구현을 작성하고 Shift+Enter로 실행합니다.
# 3) 정답과 비교한 뒤 값 하나를 바꾸고 결과를 설명합니다.
raise NotImplementedError("07번 실습의 TODO 3을 완성하세요")

In [ ]:
# TODO 4: 2. Logistic regression
# 오른쪽 정답 창의 같은 셀을 참고하되, 먼저 아래 구현을 직접 작성하세요.
# 1) 입력과 출력의 shape/dtype을 종이에 적습니다.
# 2) 최소 구현을 작성하고 Shift+Enter로 실행합니다.
# 3) 정답과 비교한 뒤 값 하나를 바꾸고 결과를 설명합니다.
raise NotImplementedError("07번 실습의 TODO 4을 완성하세요")

## 3. 왜 MLP가 필요한가: XOR

선형 층을 여러 개 이어도 activation이 없다면 하나의 선형 변환과 같습니다. XOR처럼 하나의 직선으로 나눌 수 없는 문제에는 비선형 activation이 필요합니다.

In [ ]:
xor_x = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
xor_y = torch.tensor([[0.0], [1.0], [1.0], [0.0]])

torch.manual_seed(SEED)
xor_linear = nn.Linear(2, 1)
optimizer = torch.optim.Adam(xor_linear.parameters(), lr=0.1)
for _ in range(300):
    optimizer.zero_grad(set_to_none=True)
    loss = nn.functional.binary_cross_entropy_with_logits(xor_linear(xor_x), xor_y)
    loss.backward()
    optimizer.step()

linear_predictions = xor_linear(xor_x).sigmoid().ge(0.5).float()
print("linear predictions:", linear_predictions.squeeze().tolist())
print("linear accuracy:", linear_predictions.eq(xor_y).float().mean().item())

In [ ]:
class TinyMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(2, 8),
            nn.Tanh(),
            nn.Linear(8, 1),
        )

    def forward(self, x):
        return self.network(x)


torch.manual_seed(SEED)
xor_mlp = TinyMLP()
optimizer = torch.optim.Adam(xor_mlp.parameters(), lr=0.05)
for step in range(401):
    optimizer.zero_grad(set_to_none=True)
    loss = nn.functional.binary_cross_entropy_with_logits(xor_mlp(xor_x), xor_y)
    loss.backward()
    optimizer.step()

mlp_probabilities = xor_mlp(xor_x).sigmoid()
mlp_predictions = mlp_probabilities.ge(0.5).float()
print("MLP probabilities:", mlp_probabilities.detach().squeeze().tolist())
print("MLP accuracy:", mlp_predictions.eq(xor_y).float().mean().item())
assert mlp_predictions.eq(xor_y).all()

## 4. Activation과 gradient

- ReLU: 계산이 싸고 깊은 MLP/CNN에서 널리 사용
- GELU: Transformer 계열에서 흔함
- Tanh: 출력 범위가 `[-1,1]`, RNN 내부에서 전통적으로 사용
- Sigmoid: 이진 확률이나 gate에 사용, 큰 절댓값에서 gradient 포화

In [ ]:
values = torch.linspace(-3, 3, 7)
activation_table = pd.DataFrame(
    {
        "x": values.tolist(),
        "relu": nn.functional.relu(values).tolist(),
        "gelu": nn.functional.gelu(values).tolist(),
        "tanh": torch.tanh(values).tolist(),
        "sigmoid": torch.sigmoid(values).tolist(),
    }
)
activation_table.round(4)


def activation_gradients(function, points):
    x = points.clone().requires_grad_(True)
    function(x).sum().backward()
    return x.grad


points = torch.tensor([-8.0, -2.0, 0.0, 2.0, 8.0])
gradient_table = pd.DataFrame(
    {
        "x": points.tolist(),
        "sigmoid_grad": activation_gradients(torch.sigmoid, points).tolist(),
        "tanh_grad": activation_gradients(torch.tanh, points).tolist(),
        "gelu_grad": activation_gradients(nn.functional.gelu, points).tolist(),
    }
)
gradient_table.round(6)

## 5. Initialization: 신호 크기를 보존하기

너무 작은 초기값은 신호와 gradient를 사라지게 하고, 너무 큰 값은 폭발시킵니다. Xavier는 tanh 같은 대칭 activation, Kaiming은 ReLU 계열과 잘 맞도록 분산을 조절합니다.

In [ ]:
torch.manual_seed(SEED)
input_signal = torch.randn(512, 64)


def output_std(weight_std):
    weight = torch.randn(64, 64) * weight_std
    return float((input_signal @ weight).std())


for std in [0.001, 0.02, 1 / math.sqrt(64), 1.0]:
    print(f"weight std={std:.4f} -> output std={output_std(std):.4f}")

In [ ]:
xavier_layer = nn.Linear(64, 64, bias=False)
kaiming_layer = nn.Linear(64, 64, bias=False)
nn.init.xavier_uniform_(xavier_layer.weight)
nn.init.kaiming_uniform_(kaiming_layer.weight, nonlinearity="relu")

with torch.inference_mode():
    xavier_output = torch.tanh(xavier_layer(input_signal))
    kaiming_output = torch.relu(kaiming_layer(input_signal))

print("Xavier+tanh std:", xavier_output.std().item())
print("Kaiming+ReLU std:", kaiming_output.std().item())
print("weight shapes:", xavier_layer.weight.shape, kaiming_layer.weight.shape)

## 6. Dropout: train과 eval의 동작 차이

dropout은 학습 중 일부 activation을 0으로 만들고 남은 값을 보정합니다. 평가 시에는 꺼져야 하므로 `model.eval()` 전환이 중요합니다.

In [ ]:
dropout = nn.Dropout(p=0.5)
ones = torch.ones(12)

torch.manual_seed(SEED)
dropout.train()
train_output = dropout(ones)
dropout.eval()
eval_output = dropout(ones)

print("train:", train_output.tolist())
print("eval: ", eval_output.tolist())
assert (train_output == 0).any()
assert torch.equal(eval_output, ones)

## 7. BatchNorm과 LayerNorm

`BatchNorm1d(D)`는 배치의 통계를 사용하므로 batch 구성과 train/eval 모드에 민감합니다. `LayerNorm(D)`는 각 샘플의 마지막 feature 축을 정규화하며 Transformer에서 널리 사용됩니다.

In [ ]:
normalization_input = torch.randn(8, 6) * 4 + 10  # [B, D]
batch_norm = nn.BatchNorm1d(6)
layer_norm = nn.LayerNorm(6)

batch_normalized = batch_norm(normalization_input)
layer_normalized = layer_norm(normalization_input)

print("BatchNorm feature means:", batch_normalized.mean(dim=0).round(decimals=4))
print("LayerNorm sample means: ", layer_normalized.mean(dim=1).round(decimals=4))
assert batch_normalized.shape == layer_normalized.shape == normalization_input.shape

## 8. 작은 배치 하나를 과적합하라

실제 train loss가 내려가지 않을 때 모델을 키우기 전에 8~32개 샘플만 골라 거의 완벽히 외울 수 있는지 확인합니다. 실패한다면 데이터/label, loss, gradient, optimizer 또는 모델 연결에 버그가 있을 가능성이 큽니다.

In [ ]:
torch.manual_seed(SEED)
tiny_x = torch.randn(8, 5)
tiny_y = torch.tensor([0, 2, 1, 2, 0, 1, 1, 2])
overfit_model = nn.Sequential(
    nn.Linear(5, 32), nn.GELU(), nn.Linear(32, 32), nn.GELU(), nn.Linear(32, 3)
)
optimizer = torch.optim.Adam(overfit_model.parameters(), lr=0.03)

for step in range(201):
    optimizer.zero_grad(set_to_none=True)
    logits = overfit_model(tiny_x)
    loss = nn.functional.cross_entropy(logits, tiny_y)
    loss.backward()
    optimizer.step()
    if step % 50 == 0:
        accuracy = logits.argmax(-1).eq(tiny_y).float().mean()
        print(f"step={step:3d} loss={loss.item():.5f} accuracy={accuracy.item():.2f}")

final_accuracy = overfit_model(tiny_x).argmax(-1).eq(tiny_y).float().mean()
assert final_accuracy == 1.0

In [ ]:
# gradient 상태를 한 줄씩 요약하는 도구
overfit_model.zero_grad(set_to_none=True)
diagnostic_loss = nn.functional.cross_entropy(overfit_model(tiny_x), tiny_y)
diagnostic_loss.backward()

gradient_rows = []
for name, parameter in overfit_model.named_parameters():
    gradient_rows.append(
        {
            "parameter": name,
            "shape": tuple(parameter.shape),
            "parameter_norm": float(parameter.detach().norm()),
            "gradient_norm": None
            if parameter.grad is None
            else float(parameter.grad.norm()),
            "finite": parameter.grad is not None
            and bool(torch.isfinite(parameter.grad).all()),
        }
    )
pd.DataFrame(gradient_rows)

In [ ]:
# Gradient clipping은 폭발한 gradient의 방향은 유지하고 전체 norm을 제한합니다.
clip_model = nn.Linear(3, 2)
huge_input = torch.randn(16, 3) * 10_000
clip_model(huge_input).square().mean().backward()

before_clip = torch.sqrt(sum(p.grad.square().sum() for p in clip_model.parameters()))
returned_norm = nn.utils.clip_grad_norm_(clip_model.parameters(), max_norm=1.0)
after_clip = torch.sqrt(sum(p.grad.square().sum() for p in clip_model.parameters()))
print(
    "before/returned/after:",
    before_clip.item(),
    float(returned_norm),
    after_clip.item(),
)
assert after_clip <= 1.0001

## 따라 치기 TODO + 마무리

새 셀에서 다음 실험을 진행하세요.

1. 선형 회귀의 learning rate를 `0.001, 0.08, 1.0`으로 바꾸고 loss 곡선의 차이를 기록합니다.
2. XOR MLP에서 `Tanh`를 제거해 다시 학습하고 결과를 설명합니다.
3. 10층 MLP를 만들어 initialization별로 각 층 activation std를 수집합니다.
4. 작은 배치 과적합 모델에 dropout `p=0.9`를 넣고 왜 디버깅이 어려워지는지 확인합니다.
5. `BatchNorm1d`를 batch size 1로 학습할 때 발생하는 문제를 관찰하고 `LayerNorm`과 비교합니다.
6. overfit test가 통과한 뒤 train/validation split으로 확장하고 일반화 격차를 측정합니다.

기억할 핵심: **복잡한 모델을 고치려 하지 말고, 먼저 작은 데이터에서 forward → loss → gradient → update가 모두 연결됐음을 증명하세요.**